# RotateWise: training pipeline (Colab copy)

County-level corn and soybean **yield-anomaly** model for 99 Iowa counties from NASA POWER weather. Hindcast, county scale. Development years 2000-2019 use leave-one-year-out CV; test years (2020-2025) were evaluated once by `src/final_eval.py` and are only *read* here.

## Setup
In Colab: clone the repository and install requirements first (cached data and models are in the repo).

In [1]:
# !git clone <REPO_URL> rotatewise && %cd rotatewise
# !pip install -q -r requirements.txt
import os, sys, json
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np

## 1. Data: features (B8) and yields, development years only

In [2]:
from src.cv import load_dev
from src.features import S0, S1, S2, LABELS
y, f = load_dev('corn')
print(y.year.min(), y.year.max(), len(y), 'county-years;', f.shape)
pd.Series({c: LABELS[c] for c in S2}, name='feature')

2000 2019 1955 county-years; (1980, 15)


jul_tmax_mean                 July average high temperature
precip_jun_aug                             Jun-Aug rainfall
gwetroot_aug_mean             August root-zone soil wetness
heat_days_jul_aug                  Jul-Aug days above 30 °C
water_balance_jja     Jun-Aug rain minus evapotranspiration
gwetroot_jul_mean               July root-zone soil wetness
tmin_aug_mean                      August night temperature
spring_frost_days                    late-spring frost days
heavy_rain_days_mj                  May-Jun heavy-rain days
gdd_season                    growing degree days (May-Sep)
Name: feature, dtype: str

## 2. Exploratory findings (dev years)

In [3]:
print(open('reports/eda.md').read())

# Exploratory analysis (development years 2000-2019 only)

**Findings.** 52% of corn and 66% of soybean anomaly variance is shared by all counties in the same year, so the year is the real unit of evidence (20 dev years; POWER has only about 62 distinct grid cells for 99 counties). Persistence is weak: the pooled lag-1 correlation of anomalies is 0.26 (corn) and -0.02 (soy). The worst corn years were 2012 (-26%), 2010 (-10%), 2013 (-9%). The features most correlated with corn anomalies are jul_tmax_mean (-0.30), heat_days_jul_aug (-0.26), tmin_aug_mean (-0.18), gdd_season (-0.13). Features with variance inflation above 5: {'jul_tmax_mean': np.float64(8.68), 'precip_jun_aug': np.float64(12.89), 'gwetroot_aug_mean': np.float64(17.07), 'heat_days_jul_aug': np.float64(12.66), 'water_balance_jja': np.float64(7.98), 'gwetroot_jul_mean': np.float64(17.27)}; heat, Tmax and degree days overlap, which favours regularised models. Test-period weather outside the dev range: ['heavy_rain_days_mj']. 

## 3. Pre-committed selection rule

In [4]:
print(open('reports/selection_rule.md').read())

# Pre-committed model-selection rule

Written 2026-11-14, before any cross-validation result of the revised pipeline was produced
(the first row of `experiments/log.csv` is timestamped after this file).

Evidence unit: the held-out year. Development years 2000-2019, leave-one-year-out CV,
detrending refit inside every fold. Metrics are averaged over held-out years. Corn and soybean are selected separately.

1. A candidate must beat both M0 (zero anomaly) and M2 (3-feature ridge) in mean per-year RMSE,
   and have lower RMSE than M2 in a clear majority of years (at least 12 of 20).
2. Among candidates that qualify, choose the simplest model whose mean per-year RMSE is within one
   standard error (over years) of the best. Simplicity order: M2 < M3 < M4 < M5 < M6 < M8 < M7 < ensembles.
3. If the 90% year-bootstrap interval of the paired RMSE difference versus M2 includes zero, call it a tie
   and choose the simpler model.
4. Ensembles E1/E2 and chain C1 are tried only if their trigger i

## 4. Leave-one-year-out CV: baseline, 3-feature ridge, selected model
Detrending is refit inside every fold.

In [5]:
from src.cv import loyo
from src.metrics import per_year, summary
from src.models import candidates
spec = json.load(open('models/final_spec.json'))
rows = {}
for crop in ('corn', 'soy'):
    y, f = load_dev(crop)
    c = candidates(crop)
    m0 = per_year(loyo(c['M0'], y, f))
    for k in ('M0', 'M2', spec[crop]['model_id']):
        py = per_year(loyo(c[k], y, f))
        rows[(crop, k)] = summary(py, m0)
pd.DataFrame(rows).T[['rmse', 'rmse_se', 'mae', 'skill_vs_m0', 'spearman_within_year']].round(3)

rmse  rmse_se    mae  skill_vs_m0  spearman_within_year
corn M0     11.527    1.221  9.457        0.000                   NaN
     M2     11.933    1.291  9.852       -0.035                -0.042
     M6_S1  10.735    0.928  8.640        0.069                 0.168
soy  M0     11.301    1.361  9.519        0.000                   NaN
     M2     11.648    1.326  9.842       -0.031                 0.062
     M6_S1  10.712    1.223  9.001        0.052                 0.081

## 5. Every model tried (from `experiments/log.csv`)

In [6]:
pd.read_csv('models/experiment_table.csv').round(3)

,crop,model_id,cv_rmse,rmse_90ci,cv_mae,skill_vs_m0,skill_90ci,within_year_spearman,train_rmse,selected
0,corn,M0,11.527,"[9.757, 13.708]",9.457,0.000,"[0.0, 0.0]",NaN,11.653,False
1,corn,M1,13.416,"[11.887, 15.225]",10.624,-0.164,"[-0.296, -0.057]",0.161,14.211,False
2,corn,M2,11.933,"[10.052, 14.211]",9.852,-0.035,"[-0.073, -0.002]",-0.042,11.086,False
3,corn,M3_S1,12.116,"[10.14, 14.512]",10.014,-0.051,"[-0.091, -0.011]",-0.062,11.042,False
4,corn,M3_S2,12.360,"[10.171, 14.904]",10.271,-0.072,"[-0.156, 0.003]",0.006,10.568,False
5,corn,M4_l0.2,11.702,"[9.794, 14.076]",9.648,-0.015,"[-0.036, 0.0]",-0.093,11.515,False
6,corn,M4_l0.5,11.711,"[9.783, 14.096]",9.657,-0.016,"[-0.038, 0.0]",0.043,11.585,False
7,corn,M4_l0.8,11.666,"[9.771, 14.004]",9.615,-0.012,"[-0.031, 0.0]",0.362,11.614,False
8,corn,M5,12.410,"[10.576, 14.621]",10.168,-0.077,"[-0.137, -0.024]",-0.001,10.490,False
9,corn,M6_S0,10.793,"[9.454, 12.261]",8.680,0.064,"[-0.016, 0.143]",0.150,9.852,False


## 6. Robustness: train-vs-CV gap, placebo, spatial hold-out, ablations

In [7]:
dev = json.load(open('models/dev_results.json'))
for crop in dev:
    print(crop, json.dumps(dev[crop]['checks'], indent=1))

corn {
 "train_vs_cv_gap": {
  "train_rmse": 9.8103,
  "cv_rmse": 10.7351
 },
 "placebo": {
  "shuffled_rmse_mean": 11.704,
  "runs": [
   11.729,
   11.814,
   11.569
  ],
  "m0_rmse": 11.5267,
  "real_rmse": 10.7351
 },
 "spatial_block": {
  "rmse": 10.884,
  "loyo_rmse": 10.7351
 },
 "expanding_window_2008_2019": {
  "rmse": {
   "M0": 15.319,
   "M1": 15.242,
   "M2": 15.197,
   "M3_S1": 15.245,
   "M6_S1": 15.174,
   "M7_l8d2m50r10": 15.16
  },
  "loyo_rank": [
   "M6_S1",
   "M7_l8d2m50r10",
   "M0",
   "M2",
   "M3_S1",
   "M1"
  ],
  "expanding_rank": [
   "M7_l8d2m50r10",
   "M6_S1",
   "M2",
   "M1",
   "M3_S1",
   "M0"
  ]
 },
 "ablations": {
  "A1_without_gwetroot": 10.735,
  "A2_pooled_crops": 10.737,
  "A4_climatology_anomaly_features": 11.21,
  "A3_theilsen_detrending (different target)": 10.682,
  "A1_reference_selected": 10.735,
  "A1_M2_without_gwetroot": 11.924
 },
 "intervals": {
  "conformal_90_halfwidth_pct": 18.11,
  "conformal_loyo_coverage_by_year_mean": 0.897,

## 7. Locked test years (read from the one-time run)

In [8]:
run = json.load(open('models/final_test_run.json'))
print('run at', run['timestamp'], 'git', run['git_hash'], 'rerun', run['rerun'])
m = json.load(open('models/metrics.json'))
out = {}
for crop in m:
    t = m[crop]['test_expanding_trend']
    for k in ('M0', 'M1', 'M2', 'selected'):
        out[(crop, k)] = {'rmse': t[k]['rmse'], 'rmse_90ci': t[k]['boot']['rmse'], 'mae': t[k]['mae'],
                          'skill_vs_m0': t[k].get('skill_vs_m0')}
pd.DataFrame(out).T

run at 2026-11-14T18:48:40 git no commit yet rerun False


rmse         rmse_90ci     mae skill_vs_m0
corn M0          9.219    [7.84, 10.669]  7.3119         0.0
     M1        11.9831  [10.256, 13.834]  9.5362     -0.2998
     M2         9.2626   [7.715, 10.877]  7.2701     -0.0047
     selected   8.1578    [6.658, 9.699]  6.3571      0.1151
soy  M0         9.0583   [7.964, 10.344]  7.5221         0.0
     M1        11.7665    [9.72, 13.987]  9.3751      -0.299
     M2         8.5209   [7.095, 10.193]  7.1103      0.0593
     selected   7.5528    [6.252, 9.007]  6.1373      0.1662

## 8. Where it fails: 2012 (held out in LOYO) and 2020 (derecho)

In [9]:
oof = pd.read_parquet('models/cv_oof.parquet')
tp = pd.read_parquet('models/test_predictions.parquet')
for crop in ('corn', 'soy'):
    sel = spec[crop]['model_id']
    o = oof[(oof.crop == crop) & (oof.model_id == sel) & (oof.year == 2012)]
    t = tp[(tp.crop == crop) & (tp.year == 2020)]
    print(f"{crop}: 2012 predicted {o.pred.mean():+.1f}% vs actual {o.anom_pct.mean():+.1f}% | "
          f"2020 predicted {t.pred_selected.mean():+.1f}% vs actual {t.anom_pct.mean():+.1f}%")

corn: 2012 predicted -13.4% vs actual -27.0% | 2020 predicted -1.4% vs actual -7.6%
soy: 2012 predicted -12.5% vs actual -13.4% | 2020 predicted -5.8% vs actual -5.4%


## 9. Stress profile that feeds the rotation engine

In [10]:
sp = json.load(open('models/stress_profile.json'))
pd.DataFrame({k: sp[k] for k in ('corn', 'soybean')}).round(3)

,corn,soybean
4,0.000,0.000
5,0.000,0.000
6,0.158,0.217
7,0.651,0.397
8,0.191,0.386
9,0.000,0.000
10,0.000,0.000
